# 과제 1: California Housing 데이터로 
4개의 회귀 모델(Base Models)를 합쳐서 하나의 예측값을 만드는 앙상블를 만들어서 MSE, R²로 평가
 1) VotingRegressor 
 2) StackingRegressor 

In [21]:
from sklearn.datasets import fetch_california_housing
from sklearn.datasets import load_breast_cancer

from sklearn.ensemble import VotingRegressor, StackingRegressor, GradientBoostingRegressor,RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression,LinearRegression

from sklearn.model_selection import train_test_split, GridSearchCV
import numpy as np

from sklearn.svm import SVC
from sklearn.metrics import r2_score, mean_squared_error 
from sklearn.metrics import accuracy_score, accuracy_score,roc_auc_score,confusion_matrix,classification_report


In [22]:
california = fetch_california_housing()
X = california.data 
y = california.target 

In [23]:
X_train, X_test, y_train, y_test = train_test_split (X,y,test_size=0.2, random_state=42)

In [24]:
# Base Models 4개
lin_reg = LinearRegression() # 선형회귀모델
tree_reg = DecisionTreeRegressor(max_depth=8, random_state=42) #의사결정나무 회귀모델
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1) #랜덤포레스트 
gbr_reg = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                    max_depth=3, random_state=42) 
# GradientBoosting : 여러 약한 모델을 순서대로 쌓아서, 앞의 실수를 뒤가 고치게 만드는 앙상블 기법

base_models = [
    ("lin", lin_reg),
    ("tree", tree_reg),
    ("rf", rf_reg),
    ("gbr", gbr_reg),]

voting_reg = VotingRegressor(estimators=base_models)
 # 모델들을 독립적으로 학습한 후 각 모델의 예측을 투표로 결합해 결론을 냄
stacking_reg = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),
    n_jobs=-1,
    passthrough=False)
# 여러 기본 도델에 예측을 메타 모델의 입력으로 사용해 최종 예측을 수행


# MSE, R2 평가
def evaluate_regressor(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return mse, r2 # ( , )튜플로 반환

v_mse, v_r2 = evaluate_regressor(voting_reg, X_train, y_train, X_test, y_test)
s_mse, s_r2 = evaluate_regressor(stacking_reg, X_train, y_train, X_test, y_test)

print("=== VotingRegressor 성능 ===")
print(f"MSE: {v_mse:.4f}") # 평균제곱오차 : 값이 작을수록 모델이 데이터를 잘 맞춘다고 봄
print(f"R² : {v_r2:.4f}") # 결정계수 : 1.0에 가까울수록 좋고 0.7정도면 괜찮은 모델

print("\n=== StackingRegressor 성능 ===")
print(f"MSE: {s_mse:.4f}") 
print(f"R² : {s_r2:.4f}")
print("=============================")

=== VotingRegressor 성능 ===
MSE: 0.3132
R² : 0.7610

=== StackingRegressor 성능 ===
MSE: 0.2480
R² : 0.8108


# 과제 2: Stacking 하이퍼파라미터 튜닝
## 최적의 Stacking 모델 찾기

1. level 0 :  Base models: 다양한 알고리즘으로 1차 예측값을 만든다.
 - LogisticRegression
 - SVC
 - RandomForestClassifier
2. 각 Base Model의 주요 파라미터 튜닝해보고 (튜닝 방법: GridSearchCV 또는 RandomizedSearchCV 사용)
 - LogisticRegression :C (규제 강도), penalty, solver 
 - SVC : C, gamma, kernel 
 - RandomForest : n_estimators, max_depth, min_samples_split, max_features 
3.level 1 :  Meta model: Base 모델들의 예측값(필요하면 원본 특성까지 포함)을 입력으로 받아 학습된 방식으로 최종 결합.
    : LogisticRegression 또는 RandomForestClassifier 
4. passthrough=False vs True 성능 차이 비교
 - passthrough=False : 2층 메타 모델의 입력 = 1층 모델들의 예측값만
 - passthrough=True : 2층 메타 모델의 입력 = 1층 예측값 + 원래 X 특성까지 같이
5. 최종적으로 제일 좋은 Stacking 모델 1개 뽑아서
→ 혼동행렬(Confusion Matrix) + Classification Report 출력

In [25]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [27]:
lr = LogisticRegression(max_iter=5000, random_state=42)
svc = SVC(probability=True, random_state=42)
rf  = RandomForestClassifier(random_state=42)

# Logistic Regression
param_grid_lr = {
    "C": [0.1, 1.0, 10.0], # 규제강도
    "penalty": ["l2"],
    "solver": ["liblinear"],}
grid_lr = GridSearchCV(
    lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring="accuracy", # 각 조합에 대해 accuracy 기준으로 성능 측정
    n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_

# SVC
param_grid_svc = {
    "C": [0.1, 1.0, 10.0],
    "gamma": ["scale", 0.01, 0.001]}
grid_svc = GridSearchCV(
    svc,
    param_grid=param_grid_svc,
    cv=5,
    scoring="accuracy",
    n_jobs=-1)
grid_svc.fit(X_train, y_train)
best_svc = grid_svc.best_estimator_

# RandomForest
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10]}
grid_rf = GridSearchCV(
    rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_


# Stacking에 사용할 Base Estimators 튜플 리스트 -> 이 리스트가 StackingClassifier의 estimators로 들어감.
base_estimators = [
    ("lr", best_lr),
    ("svc", best_svc),
    ("rf", best_rf),]

# Meta Model 
meta_lr = LogisticRegression(max_iter=5000, random_state=42)
#스태킹의 마지막에 쓰이는 로지스틱 회귀 (메타 모델)
meta_rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
# 랜덤포레스트를 메타 모델

# 네 가지 Stacking 조합 만들기
stack_models = []

stack_models.append((
    "Stack_LR_meta_passthrough_False",
    StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_lr,
        passthrough=False, # 메타 모델은 베이스 모델들의 예측값(또는 예측확률) 만 입력으로 받음.
        n_jobs=-1)
))

stack_models.append((
    "Stack_LR_meta_passthrough_True",
    StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_lr,
        passthrough=True, # 원본 피처 X + 베이스 모델 출력을 합쳐서 메타 모델에 넣음
        n_jobs=-1)
))

stack_models.append((
    "Stack_RF_meta_passthrough_False",
    StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_rf,
        passthrough=False,
        n_jobs=-1)
))

stack_models.append((
    "Stack_RF_meta_passthrough_True",
    StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_rf,
        passthrough=True,
        n_jobs=-1)
))


# 6. 각 Stacking 조합 성능 비교 (Accuracy, ROC-AUC)
best_model = None
best_name = None
best_score = -np.inf  # ROC-AUC 기준으로 선택

results = []  # Stacking 조합별 성능 비교 출력때 쓰려고 결과를 잠깐 모아두는 리스트


for name, model in stack_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] # 클래스 1에 대한 예측 확률 (이진 분류 가정) → ROC-AUC 계산용

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append((name, acc, auc)) # 이름과 점수를 기록해 둠

    if auc > best_score: 
        best_score = auc
        best_model = model
        best_name = name
    # 지금 AUC가 지금까지 최고보다 크다면 4가지 스태킹 모델 중에서 ROC-AUC가 가장 높은 모델 하나를 골라서
    #그 모델과 이름을 best_model, best_name에 저장

In [28]:
print("Best LR params:", grid_lr.best_params_)
# 로지스틱 회귀에서 C=10(약한규제) 
print("Best SVC params:", grid_svc.best_params_)
# 서포트 벡터에서는 gamma = 0.001 (작은값임, 구불구불하지않은 부드러운 결정경계가 더 좋은 값이 나옴)
print("Best RF params:", grid_rf.best_params_)
# 랜덤포레스트에는 트리가 많을수록,깊을수록 좋은값

Best LR params: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}
Best SVC params: {'C': 1.0, 'gamma': 0.001}
Best RF params: {'max_depth': None, 'n_estimators': 200}


In [29]:
print("\n=== Stacking 조합별 성능 비교 ===")
for name, acc, auc in results:
    print(f"{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  ROC-AUC : {auc:.4f}\n")
print("================================")
#Accuracy는 똑같고 ROC-AUC는 Stack_LR_meta_passthrough_False가 제일 잘나옴


=== Stacking 조합별 성능 비교 ===
Stack_LR_meta_passthrough_False
  Accuracy: 0.9561
  ROC-AUC : 0.9950

Stack_LR_meta_passthrough_True
  Accuracy: 0.9561
  ROC-AUC : 0.9964

Stack_RF_meta_passthrough_False
  Accuracy: 0.9561
  ROC-AUC : 0.9937

Stack_RF_meta_passthrough_True
  Accuracy: 0.9561
  ROC-AUC : 0.9967



In [15]:
print("===========BEST Stacking 모델 ===========")
print("Best model:", best_name)
print(f"Best ROC-AUC: {best_score:.4f}\n")
# ROC-AUC: 0.9967 → 거의 완벽하게 두 클래스를 구분하는 수준
# Accuracy(정확도): 0.96 → 약 96%를 정확히 분류
y_best_pred = best_model.predict(X_test)

print("=====Confusion Matrix=====")
print(confusion_matrix(y_test, y_best_pred))
# malignant인데 benign으로 예측한 3명(False Negative) -> cancer에서는 악성을 못잡아내는게 위험한 오류임
# benign인데 malignant로 예측한 2명(False Positive)
print("\n=================Classification Report=================")
print(classification_report(y_test, y_best_pred, target_names=cancer.target_names))
print("=======================================================")

===========BEST Stacking 모델 ===========
Best model: Stack_RF_meta_passthrough_True
Best ROC-AUC: 0.9967

=====Confusion Matrix=====
[[39  3]
 [ 2 70]]

=================Classification Report=================
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

